---
title: Dataset splitting
execute:
  enabled: true
---

This executable notebook compares QRT's fixed temporal, expanding, rolling, and purged splitters on bundled SPY history. `q.dataset.TimeSeriesSplit` delegates positional fold construction to scikit-learn; QRT adds aligned named partitions, roles, metadata, diagnostics, and leakage controls.

## Import dataset and sklearn APIs

The bundled dataset keeps the example offline and reproducible. We use sklearn's `StandardScaler` inside QRT's split-aware transform pipeline.

In [1]:
from tempfile import TemporaryDirectory

import pandas as pd
from sklearn.preprocessing import StandardScaler

import qrt as q

## Build a realistic aligned dataset

We use roughly two years from the end of the bundled SPY daily history. Features are backward-looking; the target is the sign of the next five-session return. `label_end_time` records each target's horizon for purging.

In [2]:
spy = q.data.datasets.load("spy").tail(520).copy()
returns = spy["close"].pct_change()
features = pd.DataFrame(
    {
        "return_1d": returns,
        "momentum_20d": spy["close"].pct_change(20),
        "volatility_20d": returns.rolling(20).std(),
        "volume_ratio_20d": spy["volume"] / spy["volume"].rolling(20).median(),
    },
    index=spy.index,
).dropna()

horizon = 5
index = features.index[:-horizon]
positions = spy.index.get_indexer(index)
labels = (
    (spy["close"].shift(-horizon) / spy["close"] - 1)
    .loc[index]
    .gt(0)
    .astype("int8")
    .rename("target")
)
metadata = pd.DataFrame(
    {
        "symbol": "SPY",
        "label_end_time": spy.index[positions + horizon],
    },
    index=index,
)
weights = pd.Series(1.0, index=index, name="sample_weight")

dataset = q.dataset.build(
    features=features.loc[index],
    labels=labels,
    sample_weights=weights,
    metadata=metadata,
)

print(f"{len(dataset):,} sessions from {dataset.index.min():%Y-%m-%d} to {dataset.index.max():%Y-%m-%d}")
dataset.X.join(dataset.y).join(dataset.metadata).head()

495 sessions from 2024-07-22 to 2026-07-13


,return_1d,momentum_20d,volatility_20d,volume_ratio_20d,target,symbol,label_end_time
datetime,,,,,,,
2024-07-22,0.010310,0.018622,0.006460,1.070037,0,SPY,2024-07-29
2024-07-23,-0.001569,0.020341,0.006413,0.853177,0,SPY,2024-07-30
2024-07-24,-0.022663,-0.006608,0.008270,1.839450,1,SPY,2024-07-31
2024-07-25,-0.005210,-0.013016,0.008333,1.490345,1,SPY,2024-08-01
2024-07-26,0.011200,-0.003532,0.008734,1.267490,0,SPY,2024-08-02


## Fixed train, validation, and test partitions

`TemporalSplit` uses inclusive index boundaries and creates the default scheme, enabling `.train`, `.validation`, and `.test` views.

In [3]:
train_end = dataset.index[int(len(dataset) * 0.60) - 1]
validation_end = dataset.index[int(len(dataset) * 0.80) - 1]

fixed = q.dataset.TemporalSplit(
    train_end=train_end,
    validation_end=validation_end,
).apply(dataset)

q.dataset.split_diagnostics(fixed, "default")

,role,rows,proportion,start,end
partition,,,,,
train,fit,297,0.6,2024-07-22,2025-09-25
validation,evaluate,99,0.2,2025-09-26,2026-02-18
test,holdout,99,0.2,2026-02-19,2026-07-13
excluded,excluded,0,0.0,NaT,NaT


## Expanding walk-forward folds

`q.dataset.TimeSeriesSplit` wraps `sklearn.model_selection.TimeSeriesSplit`. With no `max_train_size`, each fit window expands. A five-session `gap` separates training from each 42-session test window.

In [4]:
expanding_splitter = q.dataset.TimeSeriesSplit(
    n_splits=4,
    test_size=42,
    gap=5,
    name="expanding",
)
expanding = expanding_splitter.apply(fixed)

print(expanding.splits["expanding"].metadata["backend"])
q.dataset.split_diagnostics(expanding, "expanding", "fold_1")

sklearn.model_selection.TimeSeriesSplit


,role,rows,proportion,start,end
partition,,,,,
train,fit,322,0.650505,2024-07-22,2025-10-30
test,holdout,42,0.084848,2025-11-07,2026-01-08
excluded,excluded,131,0.264646,2025-10-31,2026-07-13


## Rolling walk-forward folds

Setting sklearn's `max_train_size` turns the same wrapper into a rolling-window splitter. Here every fold fits on at most 252 SPY sessions, approximately one trading year.

In [5]:
rolling_splitter = q.dataset.TimeSeriesSplit(
    n_splits=4,
    test_size=42,
    gap=5,
    max_train_size=252,
    name="rolling",
)
rolling = rolling_splitter.apply(expanding)

q.dataset.split_diagnostics(rolling, "rolling", "fold_4")

,role,rows,proportion,start,end
partition,,,,,
train,fit,252,0.509091,2025-05-02,2026-05-04
test,holdout,42,0.084848,2026-05-12,2026-07-13
excluded,excluded,201,0.406061,2024-07-22,2026-05-11


## Purging and embargo

`PurgedTimeSeriesSplit` begins with sklearn's walk-forward folds, removes fit rows whose five-session label horizon reaches the test start, and then applies an embargo. Integer embargoes count rows, which are trading sessions for this SPY index; timedelta strings are also supported.

In [6]:
purged_splitter = q.dataset.PurgedTimeSeriesSplit(
    n_splits=4,
    test_size=42,
    max_train_size=252,
    label_end="label_end_time",
    embargo=2,
)
split_dataset = purged_splitter.apply(rolling)

calendar_embargo = q.dataset.PurgedTimeSeriesSplit(
    n_splits=4,
    test_size=42,
    max_train_size=252,
    label_end="label_end_time",
    embargo="7D",
    name="calendar_embargo",
)
split_dataset = calendar_embargo.apply(split_dataset)

display(q.dataset.split_diagnostics(split_dataset, "purged_walk_forward", "fold_1"))
display(q.dataset.split_diagnostics(split_dataset, "calendar_embargo", "fold_1"))
q.dataset.audit_splits(split_dataset, "purged_walk_forward")

,role,rows,proportion,start,end
partition,,,,,
train,fit,245,0.494949,2024-11-05,2025-10-28
test,holdout,42,0.084848,2025-11-07,2026-01-08
excluded,excluded,208,0.420202,2024-07-22,2026-07-13


,role,rows,proportion,start,end
partition,,,,,
train,fit,247,0.498990,2024-11-05,2025-10-30
test,holdout,42,0.084848,2025-11-07,2026-01-08
excluded,excluded,206,0.416162,2024-07-22,2026-07-13


,fit_rows,evaluation_rows,temporally_ordered,label_horizons_clear,passed
fold,,,,,
fold_1,245,42,True,True,True
fold_2,245,42,True,True,True
fold_3,245,42,True,True,True
fold_4,245,42,True,True,True


## Fit transformations without leakage and persist the result

`q.transform.Pipeline` wraps sklearn's pipeline but selects fit rows from QRT partition roles. It transforms every row with the learned state and records where fitting occurred. Dataset persistence retains all five split schemes and aligned components.

In [7]:
pipeline = q.transform.Pipeline([("scale", StandardScaler())])
processed = pipeline.fit_transform(
    split_dataset,
    scheme="purged_walk_forward",
    fold="fold_1",
)

fit_view = processed.partition(
    "train", scheme="purged_walk_forward", fold="fold_1"
)
assert fit_view.X.mean().abs().max() < 1e-12
assert processed.y.index.equals(split_dataset.y.index)

with TemporaryDirectory() as directory:
    path = f"{directory}/spy-dataset.joblib"
    processed.save(path)
    restored = q.dataset.Dataset.load(path)
    pd.testing.assert_frame_equal(restored.X, processed.X)
    assert restored.splits.keys() == processed.splits.keys()

pd.Series(pipeline.fit_provenance_, name="fit provenance")

scheme        purged_walk_forward
fold                       fold_1
partitions               (train,)
rows                          245
start         2024-11-05 00:00:00
end           2025-10-28 00:00:00
Name: fit provenance, dtype: object

## Compare and validate every solution

Each fold keeps complete membership by assigning unused rows to `excluded`. The final checks verify coverage, disjoint partition counts, sklearn provenance, and leakage-free purged folds.

In [8]:
examples = [
    ("default", "default"),
    ("expanding", "fold_1"),
    ("rolling", "fold_1"),
    ("purged_walk_forward", "fold_1"),
    ("calendar_embargo", "fold_1"),
]

summaries = []
for scheme, fold in examples:
    diagnostics = q.dataset.split_diagnostics(split_dataset, scheme, fold)
    assert diagnostics["rows"].sum() == len(split_dataset)
    summaries.append(
        diagnostics.reset_index().assign(scheme=scheme, fold=fold)
    )

assert split_dataset.splits["expanding"].metadata["backend"] == "sklearn.model_selection.TimeSeriesSplit"
assert q.dataset.audit_splits(split_dataset, "purged_walk_forward")["passed"].all()
assert q.dataset.audit_splits(split_dataset, "calendar_embargo")["passed"].all()

pd.concat(summaries, ignore_index=True)[
    ["scheme", "fold", "partition", "role", "rows", "proportion", "start", "end"]
]

,scheme,fold,partition,role,rows,proportion,start,end
0,default,default,train,fit,297,0.600000,2024-07-22,2025-09-25
1,default,default,validation,evaluate,99,0.200000,2025-09-26,2026-02-18
2,default,default,test,holdout,99,0.200000,2026-02-19,2026-07-13
3,default,default,excluded,excluded,0,0.000000,NaT,NaT
4,expanding,fold_1,train,fit,322,0.650505,2024-07-22,2025-10-30
5,expanding,fold_1,test,holdout,42,0.084848,2025-11-07,2026-01-08
6,expanding,fold_1,excluded,excluded,131,0.264646,2025-10-31,2026-07-13
7,rolling,fold_1,train,fit,252,0.509091,2024-10-29,2025-10-30
8,rolling,fold_1,test,holdout,42,0.084848,2025-11-07,2026-01-08
9,rolling,fold_1,excluded,excluded,201,0.406061,2024-07-22,2026-07-13
